# Machine Learning — Q&A Reference

Each section asks a question, gives a concise answer, and shows a runnable code example.

**Topics covered:**
1. Fundamentals
2. Data Preprocessing
3. Classification Algorithms
4. Regression Algorithms
5. Model Evaluation — Classification
6. Model Evaluation — Regression
7. Hyperparameter Tuning & Overfitting
8. Probability & Statistics
9. Advanced Topics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer, load_diabetes, make_classification, make_regression

# Shared toy datasets used across examples
iris = load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target

cancer = load_breast_cancer(as_frame=True)
X_cancer, y_cancer = cancer.data, cancer.target

diabetes = load_diabetes(as_frame=True)
X_diab, y_diab = diabetes.data, diabetes.target

print('Iris shape:    ', X_iris.shape)
print('Cancer shape:  ', X_cancer.shape)
print('Diabetes shape:', X_diab.shape)

---
# 1. Fundamentals

## Q1: What is the difference between supervised and unsupervised learning?

**Supervised** — every training example has a label (target). The model learns to map inputs → outputs.
- Classification: label is a category (spam/not spam)
- Regression: label is a number (house price)

**Unsupervised** — no labels. The model finds structure in the data itself.
- Clustering: group similar examples (K-Means)
- Dimensionality reduction: compress features (PCA)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# --- Supervised: predict iris species (label exists) ---
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_iris, y_iris)
print('Supervised accuracy:', clf.score(X_iris, y_iris).round(3))

# --- Unsupervised: cluster iris without using labels ---
km = KMeans(n_clusters=3, random_state=42, n_init='auto')
km.fit(X_iris)
print('Unsupervised cluster labels:', np.unique(km.labels_))

## Q2: What is train-test split and why do we need it?

We hold out a portion of data the model never sees during training. This gives an honest estimate of how well the model generalises to new data.

Without it, accuracy measured on training data is optimistic — the model may have memorised it.

**`stratify=y`** preserves the class ratio in both splits — always use it for classification.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

print('Train size:', X_train.shape[0])
print('Test  size:', X_test.shape[0])

# Class proportions should be roughly equal
print('Train class ratio:', y_train.value_counts(normalize=True).round(2).to_dict())
print('Test  class ratio:', y_test.value_counts(normalize=True).round(2).to_dict())

## Q3: What is cross-validation and when should you use it?

K-fold CV splits the data into K parts. The model is trained K times — each time one fold is the test set and the rest are training. Final score = mean across K runs.

**Why:** A single train-test split can be lucky or unlucky depending on which samples land where. CV gives a more stable, less variance-prone estimate.

**When:** Use CV for model selection and hyperparameter tuning. Use a held-out test set only for final evaluation.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
scores = cross_val_score(clf, X_cancer, y_cancer, cv=5, scoring='accuracy')

print('CV scores per fold:', scores.round(3))
print(f'Mean: {scores.mean():.3f}  Std: {scores.std():.3f}')

# Visualise fold-by-fold variance
plt.figure(figsize=(7, 3))
plt.bar(range(1, 6), scores)
plt.axhline(scores.mean(), color='red', linestyle='--', label=f'Mean = {scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation')
plt.legend()
plt.tight_layout()
plt.show()

---
# 2. Data Preprocessing

## Q4: How do you handle missing values?

`SimpleImputer` fills missing values with a chosen statistic:
- `'mean'` — good for roughly symmetric numeric columns
- `'median'` — better when the column has outliers
- `'most_frequent'` — for categorical columns
- `'constant'` — fill with a fixed value

Always fit the imputer **only on training data**, then transform both train and test.

In [ ]:
from sklearn.impute import SimpleImputer

# Synthetic data with missing values
data = pd.DataFrame({
    'age':    [25, np.nan, 35, 40, np.nan],
    'salary': [50000, 60000, np.nan, 80000, 55000],
    'city':   ['NY', 'LA', np.nan, 'NY', 'LA']
})
print('Before:\n', data)

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

data[['age', 'salary']] = num_imputer.fit_transform(data[['age', 'salary']])
data[['city']] = cat_imputer.fit_transform(data[['city']])

print('\nAfter:\n', data)

## Q5: What is the difference between StandardScaler and MinMaxScaler?

| Scaler | Formula | Output range | Use when |
|---|---|---|---|
| `StandardScaler` | `(x - mean) / std` | ~[-3, 3] | Data is roughly normal; algorithms like SVM, KNN |
| `MinMaxScaler` | `(x - min) / (max - min)` | [0, 1] | Neural networks, when bounded range is required |

**Both must be fit on training data only.**  
KNN and SVM are sensitive to scale. Decision Trees and Random Forests are not.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

sample = np.array([[10], [20], [30], [1000]])  # outlier at 1000

std  = StandardScaler().fit_transform(sample)
minmax = MinMaxScaler().fit_transform(sample)

df_cmp = pd.DataFrame({'original': sample.ravel(),
                        'StandardScaler': std.ravel().round(2),
                        'MinMaxScaler': minmax.ravel().round(2)})
print(df_cmp)
# Notice: MinMaxScaler squeezes 10/20/30 near 0 because of the outlier

## Q6: What is One-Hot Encoding and when do you need it?

Most ML algorithms require numeric input. A categorical column like `['cat', 'dog', 'bird']` can't be passed as-is — encoding it as 1/2/3 would imply an order that doesn't exist.

One-hot encoding creates one binary column per category:

```
cat  → [1, 0, 0]
dog  → [0, 1, 0]
bird → [0, 0, 1]
```

Use `handle_unknown='ignore'` so unseen categories in test data don't crash the model.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

animals = pd.DataFrame({'animal': ['cat', 'dog', 'bird', 'dog', 'cat']})

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = enc.fit_transform(animals[['animal']])

result = pd.DataFrame(encoded, columns=enc.get_feature_names_out())
print(result)

## Q7: What is a sklearn Pipeline and why should you always use one?

A `Pipeline` chains preprocessing steps and a model into a single object. Benefits:

1. **No data leakage** — the scaler/imputer is fitted only on training data inside `cross_val_score` or `GridSearchCV`.
2. **One `.fit()` / `.predict()` call** — cleaner code.
3. **Deployable** — the whole pipeline can be saved and loaded as one unit.

Without a pipeline, if you scale *before* splitting, test-set statistics leak into training — your CV scores are optimistic.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=5))
])

# cross_val_score fits the scaler fresh on each train fold — no leakage
scores = cross_val_score(pipe, X_cancer, y_cancer, cv=5, scoring='accuracy')
print(f'Pipeline CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

## Q8: What is ColumnTransformer?

Real datasets have mixed types: some columns are numeric, some categorical. `ColumnTransformer` lets you apply different preprocessing to different columns, then concatenate the results.

```
numeric cols  → impute(median) → StandardScaler  ─┐
                                                    ├→ combined feature matrix
category cols → impute(mode)   → OneHotEncoder   ─┘
```

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Synthetic mixed dataset
df = pd.DataFrame({
    'age':    [25, np.nan, 35, 40, 28],
    'income': [50, 60, np.nan, 80, 55],
    'gender': ['M', 'F', 'M', np.nan, 'F'],
    'city':   ['NY', 'LA', 'NY', 'LA', 'NY']
})
y = np.array([0, 1, 0, 1, 0])

num_cols = ['age', 'income']
cat_cols = ['gender', 'city']

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('scl', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

pipe = Pipeline([('prep', preprocessor), ('model', DecisionTreeClassifier())])
pipe.fit(df, y)

transformed = preprocessor.fit_transform(df)
print('Output shape:', transformed.shape, '  (2 numeric + 2+2 one-hot columns)')

---
# 3. Classification Algorithms

## Q9: How does K-Nearest Neighbors (KNN) work?

For a new point, KNN finds the K closest training examples (by distance) and takes a majority vote among their labels.

**Key hyperparameters:**
- `n_neighbors` (K) — small K → complex boundary (risk overfitting); large K → smoother (risk underfitting)
- `weights` — `'uniform'` (all neighbours equal) vs `'distance'` (closer ones count more)

**Requires scaling** — distance is meaningless if features are on different scales.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

results = {}
for k in [1, 5, 15, 30]:
    pipe = Pipeline([('scl', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=k))])
    score = cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean()
    results[k] = round(score, 3)

print('K → CV accuracy')
for k, s in results.items():
    print(f'  K={k:2d}: {s}')

## Q10: How does a Decision Tree work?

A decision tree repeatedly splits the data on the feature + threshold that best separates the classes. At each node it asks a yes/no question (e.g. `petal_length < 2.45`). Leaf nodes hold the final predictions.

**Split criteria:**
- `gini` — measures impurity: 0 = pure node, 0.5 = maximum impurity
- `entropy` — information gain; slightly slower but often similar results

**Key hyperparameters:** `max_depth`, `min_samples_split`, `min_samples_leaf`

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
dt.fit(X_iris, y_iris)

plt.figure(figsize=(14, 5))
plot_tree(dt, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, rounded=True)
plt.title('Decision Tree (max_depth=3) on Iris')
plt.tight_layout()
plt.show()

print('Train accuracy:', dt.score(X_iris, y_iris).round(3))
print('CV accuracy:   ', cross_val_score(dt, X_iris, y_iris, cv=5).mean().round(3))

## Q11: What is Random Forest and why is it better than a single Decision Tree?

Random Forest trains many decision trees on **random subsets** of data (bagging) and random subsets of features. Predictions are made by majority vote across all trees.

**Why it's better:**
- A single deep tree overfits. Many trees averaging out their errors gives a more stable prediction.
- The feature randomness (`max_features`) ensures trees are diverse and their errors are uncorrelated.

**Key hyperparameters:** `n_estimators` (more = better, but slower), `max_depth`, `max_features`

In [ ]:
from sklearn.ensemble import RandomForestClassifier

single_tree = DecisionTreeClassifier(random_state=42)
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)

dt_scores = cross_val_score(single_tree,  X_cancer, y_cancer, cv=5)
rf_scores  = cross_val_score(random_forest, X_cancer, y_cancer, cv=5)

print(f'Single Tree CV: {dt_scores.mean():.3f} ± {dt_scores.std():.3f}')
print(f'Random Forest CV: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}')
# RF should be higher accuracy AND lower std (more stable)

## Q12: What is Logistic Regression? (Despite the name, it's a classifier.)

Logistic Regression models the **probability** of a class using the sigmoid function:

```
P(y=1 | x) = 1 / (1 + e^(-w·x))
```

If P > 0.5 → class 1, else class 0. Unlike a Decision Tree, the decision boundary is a straight line (linear).

**Hyperparameter `C`:** inverse of regularisation strength. Small C = more regularisation (simpler model).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scl', StandardScaler()),
    ('lr',  LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

scores = cross_val_score(pipe, X_cancer, y_cancer, cv=5, scoring='accuracy')
print(f'Logistic Regression CV: {scores.mean():.3f} ± {scores.std():.3f}')

# Inspect learned coefficients (after fitting on full data)
pipe.fit(X_cancer, y_cancer)
coef = pipe.named_steps['lr'].coef_[0]
top5 = pd.Series(np.abs(coef), index=X_cancer.columns).nlargest(5)
print('\nTop 5 most influential features:')
print(top5.round(3))

## Q13: What is Naive Bayes?

Naive Bayes applies Bayes' theorem with the "naive" assumption that features are **independent** given the class:

```
P(class | features) ∝ P(class) × ∏ P(feature_i | class)
```

This is almost never true in real data, but the classifier still works surprisingly well — especially for text classification.

**Variants:**
- `GaussianNB` — numeric features assumed to be normally distributed
- `MultinomialNB` — word counts (text)
- `BernoulliNB` — binary features

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
scores = cross_val_score(nb, X_cancer, y_cancer, cv=5, scoring='accuracy')
print(f'GaussianNB CV: {scores.mean():.3f} ± {scores.std():.3f}')

# Bayes' theorem example (manual)
# P(disease | positive test) given:
p_disease  = 0.01   # prevalence
p_pos_given_disease = 0.99   # sensitivity
p_pos_given_no_disease = 0.05  # false positive rate

p_pos = p_pos_given_disease * p_disease + p_pos_given_no_disease * (1 - p_disease)
p_disease_given_pos = (p_pos_given_disease * p_disease) / p_pos

print(f'\nBayes example: P(disease | positive test) = {p_disease_given_pos:.3f}')
print('Even with a 99% accurate test, only ~17% chance of disease when prevalence is 1%')

## Q14: What is Support Vector Machine (SVM)?

SVM finds the **maximum-margin hyperplane** — the decision boundary that is as far as possible from the nearest data points (support vectors) of each class.

**The kernel trick** maps data to a higher-dimensional space, allowing non-linear boundaries without explicitly computing the transformation.

**Key hyperparameters:**
- `C` — trade-off between margin width and misclassification
- `kernel` — `'linear'`, `'rbf'` (most common), `'poly'`
- `gamma` — how far the influence of each training point reaches (for rbf)

In [ ]:
from sklearn.svm import SVC

# SVM requires scaling
for kernel in ['linear', 'rbf']:
    pipe = Pipeline([
        ('scl', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=1.0, random_state=42))
    ])
    score = cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean()
    print(f'SVM ({kernel:6s}) CV accuracy: {score:.3f}')

---
# 4. Regression Algorithms

## Q15: What is Linear Regression (Ordinary Least Squares)?

OLS finds the line (or hyperplane) that minimises the sum of squared residuals:

```
Loss = Σ (y_i - ŷ_i)²
```

The closed-form solution is:
```
w = (XᵀX)⁻¹ Xᵀy
```

This has an exact answer — no iterations needed (unlike gradient descent). However it becomes slow when the number of features is very large.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print('MAE: ', mean_absolute_error(y_test, y_pred).round(2))
print('R²:  ', r2_score(y_test, y_pred).round(3))

# OLS closed form (manual, 1-feature example)
X_simple = X_diab[['bmi']].values
X_b = np.hstack([np.ones((len(X_simple), 1)), X_simple])  # add bias column
w_ols = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_diab.values
print(f'\nOLS (manual) — intercept: {w_ols[0]:.2f}, bmi coef: {w_ols[1]:.2f}')

## Q16: What is Gradient Descent?

An iterative optimisation algorithm. At each step, move the weights in the direction that decreases the loss most (the negative gradient):

```
w := w - α × ∇Loss(w)
```

Where α is the **learning rate**:
- Too small → converges slowly
- Too large → overshoots, may diverge

Used when the matrix inversion of OLS would be too expensive (many features) or when there's no closed form (neural networks).

In [ ]:
# Gradient descent for linear regression (from scratch)
X_raw = X_diab[['bmi']].values
X_norm = (X_raw - X_raw.min()) / (X_raw.max() - X_raw.min())  # normalise to [0,1]
X_b = np.hstack([np.ones((len(X_norm), 1)), X_norm])
y = y_diab.values

def mse(w, X, y):
    return np.mean((X @ w - y) ** 2)

plt.figure(figsize=(10, 4))

for lr_val, color in [(0.01, 'steelblue'), (0.1, 'green'), (0.8, 'tomato')]:
    w = np.zeros(2)
    losses = []
    for _ in range(300):
        grad = (2 / len(y)) * X_b.T @ (X_b @ w - y)
        w = w - lr_val * grad
        losses.append(mse(w, X_b, y))
    plt.plot(losses, label=f'lr={lr_val}', color=color)

plt.yscale('log')
plt.xlabel('Iteration')
plt.ylabel('MSE (log scale)')
plt.title('Gradient Descent: Effect of Learning Rate')
plt.legend()
plt.tight_layout()
plt.show()

## Q17: What is Regularisation (Ridge / Lasso)?

Regularisation adds a penalty to the loss function to prevent large coefficients (overfitting):

| Method | Penalty | Effect |
|---|---|---|
| **Ridge (L2)** | `α × Σ w²` | Shrinks all weights toward zero, keeps all features |
| **Lasso (L1)** | `α × Σ |w|` | Forces some weights exactly to zero — **feature selection** |
| **ElasticNet** | mix of L1 + L2 | Combines both |

`alpha` controls strength. Higher alpha → more regularisation → simpler model.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

for alpha in [0.01, 1.0, 10.0]:
    pipe_ridge = Pipeline([('scl', StandardScaler()), ('reg', Ridge(alpha=alpha))])
    pipe_lasso = Pipeline([('scl', StandardScaler()), ('reg', Lasso(alpha=alpha))])

    r_ridge = cross_val_score(pipe_ridge, X_diab, y_diab, cv=5, scoring='r2').mean()
    r_lasso = cross_val_score(pipe_lasso, X_diab, y_diab, cv=5, scoring='r2').mean()
    print(f'alpha={alpha:5.2f}  Ridge R²={r_ridge:.3f}  Lasso R²={r_lasso:.3f}')

# Lasso with enough alpha zeroes some coefficients entirely
lasso = Pipeline([('scl', StandardScaler()), ('reg', Lasso(alpha=1.0))])
lasso.fit(X_diab, y_diab)
coef = pd.Series(lasso.named_steps['reg'].coef_, index=X_diab.columns)
print('\nLasso coefficients (0 = feature dropped):')
print(coef.round(2))

---
# 5. Model Evaluation — Classification

## Q18: What is a Confusion Matrix?

A confusion matrix shows the **counts** of correct and incorrect predictions broken down by class:

```
                 Predicted
              Neg    Pos
Actual  Neg  [ TN    FP ]
        Pos  [ FN    TP ]
```

- **TP** True Positive: correctly predicted positive
- **TN** True Negative: correctly predicted negative
- **FP** False Positive (Type I error): predicted positive, actually negative
- **FN** False Negative (Type II error): predicted negative, actually positive

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

pipe = Pipeline([('scl', StandardScaler()),
                 ('knn', KNeighborsClassifier(n_neighbors=7))])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Malignant', 'Benign'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — KNN on Breast Cancer')
plt.tight_layout()
plt.show()

## Q19: What is Accuracy, Precision, Recall, and F1-score?

| Metric | Formula | Answers |
|---|---|---|
| **Accuracy** | (TP+TN) / all | What fraction is correct overall? |
| **Precision** | TP / (TP+FP) | Of everything I said was positive, how many actually were? |
| **Recall** | TP / (TP+FN) | Of all actual positives, how many did I catch? |
| **F1** | 2 × P×R / (P+R) | Harmonic mean — balance between precision and recall |

**When accuracy is misleading:** If 95% of emails are not spam, a model that always predicts "not spam" gets 95% accuracy but never catches spam. Precision/recall tell the real story.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'F1 score:  {f1_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

## Q20: What is ROC-AUC and how do you interpret it?

The **ROC curve** plots True Positive Rate vs False Positive Rate at every possible classification threshold (0 → 1).

**AUC (Area Under Curve):**
- 1.0 = perfect classifier
- 0.5 = random guessing (diagonal line)
- < 0.5 = worse than random

AUC is **threshold-independent** — it tells you how well the model ranks positives above negatives, regardless of where you set the cutoff. Useful when class imbalance makes accuracy misleading.

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay

# Need predict_proba for ROC
pipe_proba = Pipeline([('scl', StandardScaler()),
                       ('knn', KNeighborsClassifier(n_neighbors=7))])
pipe_proba.fit(X_train, y_train)
y_proba = pipe_proba.predict_proba(X_test)[:, 1]

print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}')

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title('ROC Curve — KNN on Breast Cancer')
plt.tight_layout()
plt.show()

---
# 6. Model Evaluation — Regression

## Q21: What is MAE, MSE, and RMSE?

| Metric | Formula | Properties |
|---|---|---|
| **MAE** | mean(\|y - ŷ\|) | Same units as target; robust to outliers |
| **MSE** | mean((y - ŷ)²) | Penalises large errors heavily; not in original units |
| **RMSE** | √MSE | Same units as target; also penalises large errors |

Choose **MAE** when outliers shouldn't be penalised heavily (e.g. house prices). Choose **RMSE** when large errors are especially costly.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

X_tr, X_te, y_tr, y_te = train_test_split(X_diab, y_diab, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
y_hat = rf.predict(X_te)

mae  = mean_absolute_error(y_te, y_hat)
mse  = mean_squared_error(y_te, y_hat)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_hat)

print(f'MAE:  {mae:.2f}')
print(f'MSE:  {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.3f}')

plt.scatter(y_te, y_hat, alpha=0.5)
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', label='Perfect')
plt.xlabel('True')
plt.ylabel('Predicted')
plt.title('True vs Predicted — Random Forest Regression')
plt.legend()
plt.tight_layout()
plt.show()

## Q22: What is R² (coefficient of determination)?

R² measures what proportion of variance in the target is explained by the model:

```
R² = 1 - (SS_res / SS_tot)
   = 1 - Σ(y-ŷ)² / Σ(y-mean(y))²
```

- R² = 1.0 → perfect predictions
- R² = 0.0 → model does no better than predicting the mean
- R² < 0  → model is worse than predicting the mean

Unlike RMSE, R² is **scale-independent**, making it easy to compare across datasets.

In [ ]:
y_true = np.array([3, -0.5, 2, 7])
y_pred_good  = np.array([2.5,  0.0, 2, 8])
y_pred_mean  = np.full_like(y_true, y_true.mean(), dtype=float)  # always predict mean
y_pred_bad   = np.array([10,  10,  10, 10])  # terrible model

for label, yp in [('Good model', y_pred_good),
                   ('Mean baseline', y_pred_mean),
                   ('Bad model', y_pred_bad)]:
    print(f'{label:15s}  R² = {r2_score(y_true, yp):.3f}')

---
# 7. Hyperparameter Tuning & Overfitting

## Q23: What is the difference between parameters and hyperparameters?

| | Parameters | Hyperparameters |
|---|---|---|
| **What** | Values the model learns from data | Values you choose before training |
| **Examples** | Decision tree split thresholds, linear regression weights | `max_depth`, `n_neighbors`, `C`, `alpha` |
| **How set** | Automatically during `.fit()` | Manually or via grid search |

Hyperparameters control model complexity and learning behaviour. Choosing them poorly causes underfitting or overfitting.

In [ ]:
# After training, we can inspect the learned parameters
lr = LinearRegression()
lr.fit(X_diab, y_diab)

print('Learned parameters (coefficients):')
for feat, coef in zip(X_diab.columns, lr.coef_):
    print(f'  {feat:4s}: {coef:8.2f}')
print(f'  bias (intercept): {lr.intercept_:.2f}')
print()
print('Hyperparameters (you control these, not learned):')
print(' ', lr.get_params())

## Q24: What is overfitting and underfitting?

```
                      Model Complexity →
Error
  │\                              /
  │  \   Underfitting            / Overfitting
  │    \_________________________/
  │              Sweet spot
  └──────────────────────────────────
```

- **Underfitting (high bias):** model too simple → fails on both train and test
- **Overfitting (high variance):** model too complex → great on train, poor on test
- **Signals:** large gap between train accuracy and CV/test accuracy = overfitting

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

depths = range(1, 20)
train_acc, test_acc, cv_acc = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_acc.append(accuracy_score(y_tr, dt.predict(X_tr)))
    test_acc.append(accuracy_score(y_te, dt.predict(X_te)))
    cv_acc.append(cross_val_score(dt, X_cancer, y_cancer, cv=5).mean())

best_d = list(depths)[np.argmax(cv_acc)]

plt.figure(figsize=(11, 5))
plt.plot(depths, train_acc, marker='o', label='Train', color='steelblue')
plt.plot(depths, test_acc,  marker='o', label='Test',  color='tomato')
plt.plot(depths, cv_acc,    marker='o', label='CV (5-fold)', color='green', linestyle='--')
plt.axvline(best_d, color='gray', linestyle=':', linewidth=1.5)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Underfitting vs Overfitting — Decision Tree')
plt.legend()
plt.xticks(list(depths))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Best depth by CV: {best_d}')

## Q25: What is GridSearchCV and how does it work?

GridSearchCV exhaustively tries every combination of hyperparameters from a grid and evaluates each using cross-validation. It returns the combination with the best CV score.

**Naming inside a Pipeline:** use `stepname__param` — e.g. `'model__max_depth'`.

**Watch out:** with a large grid, the number of fits = `|combinations| × cv_folds`. 100 combinations × 5 folds = 500 model fits. Use `n_jobs=-1` to parallelise.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('scl',   StandardScaler()),
    ('model', KNeighborsClassifier())
])

param_grid = {
    'model__n_neighbors': [3, 5, 7, 11, 15, 21],
    'model__weights':     ['uniform', 'distance']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_cancer, y_cancer)

print('Best params: ', grid.best_params_)
print('Best CV score:', round(grid.best_score_, 4))

## Q26: What is RandomizedSearchCV and when should you use it instead of GridSearchCV?

GridSearchCV tries **every combination**. With large grids this is expensive. RandomizedSearchCV samples a **random subset** of parameter combinations — you control how many via `n_iter`.

**Rule of thumb:** use `GridSearchCV` for small grids (< ~100 combos); use `RandomizedSearchCV` for large grids or when parameters span wide continuous ranges.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

pipe = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

param_dist = {
    'model__n_estimators':      randint(50, 300),
    'model__max_depth':         [None, 3, 5, 7, 10],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf':  randint(1, 10)
}

rand_search = RandomizedSearchCV(
    pipe, param_dist, n_iter=20, cv=5,
    scoring='accuracy', random_state=42, n_jobs=-1
)
rand_search.fit(X_cancer, y_cancer)

print('Best params: ', rand_search.best_params_)
print('Best CV score:', round(rand_search.best_score_, 4))

---
# 8. Probability & Statistics

## Q27: What is a PMF, PDF, and CDF?

| | PMF | PDF | CDF |
|---|---|---|---|
| **Used for** | Discrete variables | Continuous variables | Both |
| **Gives** | P(X = x) | Density at x (not probability) | P(X ≤ x) |
| **Integrates to** | 1 (sum) | 1 (integral) | Reaches 1 at the right |

Example: rolling a die → PMF.  Height of a person → PDF.

In [ ]:
from scipy.stats import norm

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# PMF — fair die
faces = np.arange(1, 7)
axes[0].bar(faces, [1/6]*6, color='steelblue', edgecolor='white')
axes[0].set_title('PMF — Fair Die')
axes[0].set_xlabel('Outcome')
axes[0].set_ylabel('P(X = x)')

# PDF — standard normal
x = np.linspace(-4, 4, 300)
axes[1].plot(x, norm.pdf(x), color='tomato', linewidth=2)
axes[1].fill_between(x, norm.pdf(x), alpha=0.2, color='tomato')
axes[1].set_title('PDF — Standard Normal')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Density')

# CDF — standard normal
axes[2].plot(x, norm.cdf(x), color='green', linewidth=2)
axes[2].axhline(0.5, color='gray', linestyle='--', label='P(X ≤ 0) = 0.5')
axes[2].set_title('CDF — Standard Normal')
axes[2].set_xlabel('x')
axes[2].set_ylabel('P(X ≤ x)')
axes[2].legend()

plt.tight_layout()
plt.show()

## Q28: What is Bayes' Theorem?

Bayes' theorem updates the probability of a hypothesis given new evidence:

```
P(H | E) = P(E | H) × P(H) / P(E)
```

- **P(H)** — prior: what you believed before seeing evidence
- **P(E|H)** — likelihood: probability of evidence if hypothesis is true
- **P(H|E)** — posterior: updated belief after seeing evidence

This is the foundation of Naive Bayes classifiers and probabilistic ML.

In [ ]:
# Medical test example
scenarios = [
    ('Rare disease (1%)',    0.01),
    ('Common disease (10%)', 0.10),
    ('Very common (40%)',    0.40),
]

sensitivity = 0.99   # P(test+ | disease)
fpr         = 0.05   # P(test+ | no disease)

print(f'Test sensitivity: {sensitivity*100:.0f}%  False positive rate: {fpr*100:.0f}%')
print('-' * 55)

for label, prevalence in scenarios:
    p_positive = sensitivity * prevalence + fpr * (1 - prevalence)
    posterior  = (sensitivity * prevalence) / p_positive
    print(f'{label:25s} → P(disease | positive) = {posterior:.2%}')

## Q29: What is Maximum Likelihood Estimation (MLE)?

MLE finds the parameter values that make the observed data most probable:

```
θ_MLE = argmax_θ  P(data | θ)
```

In practice we maximise the **log-likelihood** (easier to compute, same answer).

Example: estimating the mean and std of a Gaussian from data. MLE gives `μ = sample_mean`, `σ = sample_std`.

In [ ]:
from scipy.stats import norm
from scipy.optimize import minimize

# Generate data from N(5, 2)
np.random.seed(42)
data = np.random.normal(loc=5, scale=2, size=200)

def neg_log_likelihood(params, data):
    mu, sigma = params
    if sigma <= 0:
        return np.inf
    return -np.sum(norm.logpdf(data, loc=mu, scale=sigma))

result = minimize(neg_log_likelihood, x0=[0, 1], args=(data,))
mu_mle, sigma_mle = result.x

print(f'True    μ=5, σ=2')
print(f'MLE     μ={mu_mle:.3f}, σ={sigma_mle:.3f}')
print(f'Sample  μ={data.mean():.3f}, σ={data.std():.3f}')
# MLE = sample mean / sample std for Gaussian

---
# 9. Advanced Topics

## Q30: What is PCA (Principal Component Analysis)?

PCA finds the directions (principal components) of maximum variance in the data and projects data onto them. It:

1. **Reduces dimensionality** — keep top K components instead of all features
2. **Removes correlated features** — components are orthogonal by construction
3. **Helps visualisation** — project to 2D to see clusters

`explained_variance_ratio_` tells you how much variance each component captures.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cancer)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
print(f'Together: {pca.explained_variance_ratio_.sum():.1%} of variance')

plt.figure(figsize=(7, 5))
for cls, color, label in [(0, 'tomato', 'Malignant'), (1, 'steelblue', 'Benign')]:
    mask = y_cancer == cls
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label, alpha=0.6, s=20)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Breast Cancer — PCA to 2D')
plt.legend()
plt.tight_layout()
plt.show()

## Q31: What is Feature Importance and how do you use it?

Tree-based models (Decision Tree, Random Forest) compute **feature importance** — how much each feature contributes to reducing impurity across all splits.

This answers: *which features actually matter?*

Uses:
- Remove unimportant features to simplify the model
- Understand what drives predictions
- Detect data leakage (suspiciously high importance)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_cancer, y_cancer)

importances = pd.Series(rf.feature_importances_, index=X_cancer.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 8))
importances.tail(15).plot(kind='barh', color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

## Q32: What is K-Means clustering?

K-Means partitions data into K clusters by iterating:
1. Assign each point to the nearest centroid
2. Recompute centroids as the mean of assigned points
3. Repeat until centroids stop moving

**Choosing K:** use the **elbow method** — plot inertia (sum of squared distances to nearest centroid) vs K and look for where the decrease flattens.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X_iris)

inertias = []
ks = range(1, 10)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(ks, inertias, marker='o', color='steelblue')
plt.axvline(3, color='tomato', linestyle='--', label='True K=3')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow Method — Iris Dataset')
plt.legend()
plt.tight_layout()
plt.show()

## Q33: How do you compare multiple models systematically?

Run the same cross-validation setup on all models and compare mean ± std. Use a held-out test set only once at the very end on the winner.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    'KNN (k=7)':          Pipeline([('scl', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=7))]),
    'Decision Tree':      DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':      RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression':Pipeline([('scl', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'Naive Bayes':        GaussianNB(),
    'SVM (rbf)':          Pipeline([('scl', StandardScaler()), ('m', SVC(kernel='rbf', random_state=42))]),
}

print(f'{"Model":25s}  {"CV Mean":>8}  {"CV Std":>7}')
print('-' * 45)
for name, model in models.items():
    s = cross_val_score(model, X_cancer, y_cancer, cv=5, scoring='accuracy')
    print(f'{name:25s}  {s.mean():.4f}    ±{s.std():.4f}')

## Q34: What is data leakage and how do you prevent it?

Data leakage happens when information from the test set (or future data) influences training — making model performance look better than it actually is.

**Common sources:**
1. Fitting a scaler on all data before splitting
2. Imputing missing values with statistics from the full dataset
3. Feature engineering using target values
4. Time series: using future data to predict the past

**Prevention: always use a Pipeline.** `cross_val_score(pipeline, ...)` fits preprocessing fresh on each fold's training data.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# WRONG: scale before CV — test fold statistics leak into training
X_leaked = StandardScaler().fit_transform(X_cancer)   # fit on ALL data first
score_leaked = cross_val_score(KNeighborsClassifier(n_neighbors=7),
                               X_leaked, y_cancer, cv=5).mean()

# CORRECT: scaler inside pipeline — fitted only on each fold's train split
pipe = Pipeline([('scl', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=7))])
score_correct = cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean()

print(f'Leaked   CV accuracy: {score_leaked:.4f}')
print(f'Correct  CV accuracy: {score_correct:.4f}')
print('(On large, clean datasets the difference is small but the principle is critical)')

---
## Summary Table

| Topic | Key concept | Covered in |
|---|---|---|
| Train-test split | Honest evaluation | Q2 |
| Cross-validation | Stable score estimate | Q3 |
| Missing values | SimpleImputer | Q4 |
| Scaling | StandardScaler / MinMaxScaler | Q5 |
| Encoding | OneHotEncoder | Q6 |
| Pipeline | Prevent leakage | Q7, Q34 |
| ColumnTransformer | Mixed data types | Q8 |
| KNN | Distance-based voting | Q9 |
| Decision Tree | Recursive splitting | Q10 |
| Random Forest | Ensemble / bagging | Q11 |
| Logistic Regression | Sigmoid / linear boundary | Q12 |
| Naive Bayes | Bayes + independence | Q13 |
| SVM | Max-margin hyperplane | Q14 |
| Linear Regression (OLS) | Closed-form least squares | Q15 |
| Gradient Descent | Iterative optimisation | Q16 |
| Regularisation (L1/L2) | Ridge, Lasso | Q17 |
| Confusion Matrix | TP/TN/FP/FN | Q18 |
| Accuracy/Precision/Recall/F1 | Classification metrics | Q19 |
| ROC-AUC | Threshold-free ranking | Q20 |
| MAE / MSE / RMSE | Regression error | Q21 |
| R² | Explained variance | Q22 |
| Parameters vs Hyperparameters | What the model learns vs what you set | Q23 |
| Overfitting / Underfitting | Bias-variance tradeoff | Q24 |
| GridSearchCV | Exhaustive tuning | Q25 |
| RandomizedSearchCV | Efficient large-grid tuning | Q26 |
| PMF / PDF / CDF | Probability distributions | Q27 |
| Bayes' Theorem | Prior → posterior | Q28 |
| MLE | Fit distributions to data | Q29 |
| PCA | Dimensionality reduction | Q30 |
| Feature Importance | Which features matter | Q31 |
| K-Means | Unsupervised clustering | Q32 |
| Model comparison | CV across all models | Q33 |
| Data leakage | Pipeline prevents it | Q34 |